# 🍐 Little Fig — Train LLMs on Any Hardware

**7× faster than BnB NF4 on GPU | Beats NF4 quality on 156/156 layers | 8GB RAM training on CPU**

| Research Finding | Improvement |
|---|---|
| FigMeZO (inverse error shaping) | −18.6% loss vs standard MeZO |
| Sensitivity-guided LISA | −10% loss vs random layer selection |
| GPU training speed | 7× faster than BnB NF4 QLoRA |
| Quantization quality | Wins 156/156 TinyLlama layers vs NF4 |

**Author:** 0xticketguy / Harboria Labs | **License:** Mixed; see NOTICE.md

[![GitHub](https://img.shields.io/badge/GitHub-littlefig-black)](https://github.com/ticketguy/littlefig)

In [ ]:
# Install (takes ~2 min)
!pip install -q torch
!pip install -q git+https://github.com/ticketguy/littlefig.git#egg=little-fig[train]

import torch
print(f'✅ Installed | PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name()}')

## 1. Quick Start: Fine-tune TinyLlama in 5 Minutes

In [ ]:
from little_fig.engine import FigModel, FigTrainer, FigTrainingConfig
from little_fig.engine.tier import TrainingTier

# Load TinyLlama with FigQuant INT4 + LoRA
model = FigModel.from_pretrained(
    'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    lora_r=16,
    lora_alpha=32,
    shared_codebook=True,  # 5× faster loading
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
# Configure and train
config = FigTrainingConfig(
    num_epochs=1,
    learning_rate=2e-4,
    max_seq_length=256,  # shorter for Colab speed
    batch_size=2,
    gradient_accumulation_steps=4,
    logging_steps=5,
    use_packing=True,
)

trainer = FigTrainer(model, config)
trainer.load_dataset('tatsu-lab/alpaca', max_samples=200)
trainer.train()

# Save (only ~5MB for the adapter)
model.save_adapter('./my_adapter')

## 2. Memory Fabric — The Model Remembers

Memory lives IN the model weights. No external database. No RAG.

In [ ]:
# Load with Memory Fabric
model = FigModel.from_pretrained(
    'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    lora_r=16,
    memory_fabric=True,
    shared_codebook=True,
)

# Write memories INTO the weights
r1 = model.write_memory('personal', 'User prefers Python for backend work.')
r2 = model.write_memory('wiki', 'Speed of light is 299,792,458 m/s.')
r3 = model.write_memory('schedule', 'Team standup every day at 9:15am.')

print(f'Memory written in {r1["time_ms"]:.0f}ms')
print(f'\nMemory confidence per namespace:')
for ns, info in model.memory_confidence().items():
    if info['mean_magnitude'] > 0:
        print(f'  {ns}: {info["mean_magnitude"]:.4f}')

## 3. FigMeZO — Train Without Backward Passes

Original research: −18.6% loss vs standard MeZO.
Uses only forward passes — fits in inference-level memory.

In [ ]:
from little_fig.engine.figmezo import FigMeZO, FigMeZOConfig

# MeZO: gradient-free training (only forward passes!)
optimizer = FigMeZO(model.model, FigMeZOConfig(
    learning_rate=1e-5,
    epsilon=1e-3,
    shaping_strength=-0.3,  # Negative = our novel inverse shaping
))

# Each step uses 2 forward passes, 0 backward passes
import torch
model.model.eval()
for step in range(5):
    ids = torch.randint(0, 32000, (1, 32))
    if torch.cuda.is_available(): ids = ids.cuda()
    loss = optimizer.step(lambda: model(input_ids=ids, labels=ids).loss)
    print(f'  Step {step}: loss={loss:.4f}')

## 4. Benchmark Results

All results validated on Tesla T4 GPU with TinyLlama 1.1B.

### Quantization Quality (156 layers)
| Method | MSE | Cosine | Wins |
|---|---|---|---|
| **FigQuant** | **5.64e-6** | **0.9956** | **156/156** |
| NF4 (QLoRA) | 5.97e-6 | 0.9953 | 0/156 |

### Training Speed
| Method | Loss | Time | Speed |
|---|---|---|---|
| FP16 LoRA | 0.2252 | 1309s | 1× |
| BnB NF4 | 0.2399 | 1423s | 0.9× |
| **FigQuant** | **0.2475** | **184s** | **7×** |

---
*Built by 0xticketguy / Harboria Labs*
*License: Mixed; see NOTICE.md*